In [1]:
!pip install jiwer
!pip install evaluate
!pip install pytorch_lightning

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 29.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 3.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 857.3/857.3 kB 14.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 983.4/983.4 kB 50.3 MB/s eta 0:00:00


In [2]:
!pip install gdown

In [3]:
!gdown 1j9d91QqE7_WnOnmEmidtOG55tpmxQUeJ

Downloading...
From (original): https://drive.google.com/uc?id=1j9d91QqE7_WnOnmEmidtOG55tpmxQUeJ
From (redirected): https://drive.google.com/uc?id=1j9d91QqE7_WnOnmEmidtOG55tpmxQUeJ&confirm=t&uuid=522aaba6-0167-482c-b6a2-1090bdf7cf77
To: /content/dataset.zip
100% 9.12G/9.12G [02:06<00:00, 72.2MB/s]


In [4]:
!unzip dataset.zip -d dataset

Streaming output truncated to the last 5000 lines.
  inflating: dataset/toronto_6/toronto_6_38.wav  
  inflating: dataset/toronto_6/toronto_6_39.wav  
  inflating: dataset/toronto_6/toronto_6_40.wav  
  inflating: dataset/toronto_6/toronto_6_41.wav  
  inflating: dataset/toronto_6/toronto_6_42.wav  
  inflating: dataset/toronto_6/toronto_6_43.wav  
  inflating: dataset/toronto_6/toronto_6_44.wav  
  inflating: dataset/toronto_6/toronto_6_45.wav  
  inflating: dataset/toronto_6/toronto_6_46.wav  
  inflating: dataset/toronto_6/toronto_6_47.wav  
  inflating: dataset/toronto_6/toronto_6_48.wav  
  inflating: dataset/toronto_6/toronto_6_49.wav  
  inflating: dataset/toronto_6/toronto_6_50.wav  
  inflating: dataset/toronto_6/toronto_6_51.wav  
  inflating: dataset/toronto_6/toronto_6_52.wav  
  inflating: dataset/toronto_6/toronto_6_53.wav  
  inflating: dataset/toronto_6/toronto_6_54.wav  
  inflating: dataset/toronto_6/toronto_6_55.wav  
  inflating: dataset/toronto_6/toronto_6_56.wav  

In [5]:
!ls /content/dataset

labels.jsonl  toronto_139  toronto_166	toronto_35  toronto_55	toronto_81
toronto_0     toronto_14   toronto_17	toronto_36  toronto_58	toronto_83
toronto_100   toronto_144  toronto_170	toronto_37  toronto_59	toronto_84
toronto_101   toronto_145  toronto_172	toronto_38  toronto_6	toronto_85
toronto_11    toronto_148  toronto_18	toronto_4   toronto_60	toronto_86
toronto_12    toronto_15   toronto_187	toronto_42  toronto_62	toronto_87
toronto_123   toronto_150  toronto_188	toronto_43  toronto_66	toronto_89
toronto_127   toronto_153  toronto_2	toronto_44  toronto_67	toronto_9
toronto_128   toronto_155  toronto_21	toronto_45  toronto_68	toronto_92
toronto_130   toronto_156  toronto_23	toronto_46  toronto_7	toronto_93
toronto_133   toronto_157  toronto_25	toronto_49  toronto_72	toronto_94
toronto_134   toronto_159  toronto_26	toronto_5   toronto_74	toronto_95
toronto_135   toronto_16   toronto_27	toronto_50  toronto_75	toronto_97
toronto_136   toronto_161  toronto_3	toronto_53  toronto_77
tor

In [6]:
import os
import re
import json
import random
import logging
import evaluate
import torch
import torchaudio
import pytorch_lightning as pl

from tqdm import tqdm
from pathlib import Path
from dataclasses import dataclass
from pytorch_lightning import Trainer
from torch.utils.data import Dataset, DataLoader
from transformers import WhisperProcessor, WhisperForConditionalGeneration

In [7]:
logger = logging.getLogger(__name__)
logger.setLevel(logging.INFO)

In [8]:
AUDIO_DIR = Path("/content")

TEST_IDS = [
    "toronto_27", "toronto_46", "toronto_42", "toronto_37",
    "toronto_43", "toronto_157", "toronto_9", "toronto_156",
    "toronto_7", "toronto_123", "toronto_54", "toronto_67",
    "toronto_62", "toronto_81", "toronto_134", "toronto_148",
    "toronto_21", "toronto_135", "toronto_166", "toronto_58",
]

In [9]:
def split_data(items):
    train, val, test = [], [], []

    for item in items:
        file_id = item["path"].split("/")[-2]  # toronto_157

        if file_id in TEST_IDS:
            test.append(item)
        else:
            train.append(item)

    random.shuffle(train)
    val = train[:int(0.05 * len(train))]
    train = train[int(0.05 * len(train)):]

    return train, val, test

In [10]:
def normalize_text(text):
    text = text.lower()
    text = re.sub(r"[^\w\sа-яіїєґ]", "", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text

In [11]:
def load_labels(path):
    with open(path, "r", encoding="utf-8") as f:
        data = json.load(f)

    items = []
    missing = 0

    for audio_path, text in data.items():
        if os.path.exists(AUDIO_DIR / audio_path):
            items.append({
                "path": audio_path,
                "text": text
            })
        else:
            missing += 1

    print(f"Loaded: {len(items)}")
    print(f"Missing files skipped: {missing}")

    return items

In [12]:
class TorontoDataset(Dataset):
    def __init__(self, items, processor):
        self.items = items
        self.processor = processor

    def __len__(self):
        return len(self.items)

    def __getitem__(self, idx):
        item = self.items[idx]

        audio, sr = torchaudio.load(AUDIO_DIR / item["path"])
        audio = audio.mean(dim=0)  # mono

        if sr != 16000:
            audio = torchaudio.functional.resample(audio, sr, 16000)

        text = normalize_text(item["text"])

        inputs = self.processor(
            audio,
            sampling_rate=16000,
            return_tensors="pt",
        )

        labels = self.processor(
            text=text,
            return_tensors="pt"
        ).input_ids

        return {
            "input_features": inputs.input_features[0],
            "labels": labels[0]
        }

In [13]:
@dataclass
class DataCollator:
    processor: any

    def __call__(self, batch):
        input_features = [b["input_features"] for b in batch]
        labels = [b["labels"] for b in batch]

        input_features = torch.nn.utils.rnn.pad_sequence(
            input_features, batch_first=True
        )

        attention_mask = torch.ones_like(input_features[:, :, 0])

        labels = torch.nn.utils.rnn.pad_sequence(
            labels, batch_first=True, padding_value=-100
        )

        return {
            "input_features": input_features,
            "attention_mask": attention_mask,
            "labels": labels
        }

In [14]:
class WhisperLightning(pl.LightningModule):
    def __init__(self, model_name, processor, lr=1e-5):
        super().__init__()
        self.model = WhisperForConditionalGeneration.from_pretrained(model_name)

        self.model.generation_config.suppress_tokens = []
        self.model.generation_config.forced_decoder_ids = None

        self.processor = processor
        self.lr = lr

    def training_step(self, batch, batch_idx):
        out = self.model(**batch)
        self.log("train_loss", out.loss, prog_bar=True, on_step=True, on_epoch=True)
        return out.loss

    def validation_step(self, batch, batch_idx):
        out = self.model(**batch)
        self.log("val_loss", out.loss, prog_bar=True, on_step=False, on_epoch=True)

    def configure_optimizers(self):
        return torch.optim.AdamW(self.parameters(), lr=self.lr)

In [15]:
processor = WhisperProcessor.from_pretrained(
    "openai/whisper-tiny",
    language="uk",
    task="transcribe",
)

items = load_labels(AUDIO_DIR / "dataset/labels.jsonl")
train_items, val_items, test_items = split_data(items)

collator = DataCollator(processor)

model = WhisperLightning("openai/whisper-tiny", processor)
model.train()

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


preprocessor_config.json: 0.00B [00:00, ?B/s]

config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

...

Loaded: 18303
Missing files skipped: 10929


model.safetensors:   0%|          | 0.00/151M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/167 [00:00<?, ?it/s]

generation_config.json: 0.00B [00:00, ?B/s]

WhisperLightning(
  (model): WhisperForConditionalGeneration(
    (model): WhisperModel(
      (encoder): WhisperEncoder(
        (conv1): Conv1d(80, 384, kernel_size=(3,), stride=(1,), padding=(1,))
        (conv2): Conv1d(384, 384, kernel_size=(3,), stride=(2,), padding=(1,))
        (embed_positions): Embedding(1500, 384)
        (layers): ModuleList(
          (0-3): 4 x WhisperEncoderLayer(
            (self_attn): WhisperAttention(
              (k_proj): Linear(in_features=384, out_features=384, bias=False)
              (v_proj): Linear(in_features=384, out_features=384, bias=True)
              (q_proj): Linear(in_features=384, out_features=384, bias=True)
              (out_proj): Linear(in_features=384, out_features=384, bias=True)
            )
            (self_attn_layer_norm): LayerNorm((384,), eps=1e-05, elementwise_affine=True)
            (activation_fn): GELUActivation()
            (fc1): Linear(in_features=384, out_features=1536, bias=True)
            (fc2): Linea

In [16]:
wer_metric = evaluate.load("wer")
cer_metric = evaluate.load("cer")

def evaluate_model(model, dataloader, processor):
    preds, refs = [], []

    model.eval()
    for batch in tqdm(dataloader):
        with torch.no_grad():
            pred_ids = model.model.generate(
                batch["input_features"].to(model.device),
                language="uk",
                task="transcribe",
            )

        pred_str = processor.batch_decode(pred_ids, skip_special_tokens=True)

        labels = batch["labels"].clone()
        labels[labels == -100] = processor.tokenizer.pad_token_id
        ref_str = processor.batch_decode(labels, skip_special_tokens=True)

        preds.extend(pred_str)
        refs.extend(ref_str)

    return preds, refs

In [17]:
next(model.model.parameters()).device

device(type='cpu')

In [18]:
model.to("cuda")

WhisperLightning(
  (model): WhisperForConditionalGeneration(
    (model): WhisperModel(
      (encoder): WhisperEncoder(
        (conv1): Conv1d(80, 384, kernel_size=(3,), stride=(1,), padding=(1,))
        (conv2): Conv1d(384, 384, kernel_size=(3,), stride=(2,), padding=(1,))
        (embed_positions): Embedding(1500, 384)
        (layers): ModuleList(
          (0-3): 4 x WhisperEncoderLayer(
            (self_attn): WhisperAttention(
              (k_proj): Linear(in_features=384, out_features=384, bias=False)
              (v_proj): Linear(in_features=384, out_features=384, bias=True)
              (q_proj): Linear(in_features=384, out_features=384, bias=True)
              (out_proj): Linear(in_features=384, out_features=384, bias=True)
            )
            (self_attn_layer_norm): LayerNorm((384,), eps=1e-05, elementwise_affine=True)
            (activation_fn): GELUActivation()
            (fc1): Linear(in_features=384, out_features=1536, bias=True)
            (fc2): Linea

In [20]:
test_ds = TorontoDataset(test_items, processor)
test_loader = DataLoader(
    test_ds, batch_size=8, num_workers=4, collate_fn=collator,
)

preds, refs = evaluate_model(model, test_loader, processor)

indices = random.sample(range(len(preds)), 10)
print("Sample predictions:")
for idx in indices:
    print(f"Ref: {refs[idx]}")
    print(f"Pred: {preds[idx]}")
    print("-" * 20)

wer = wer_metric.compute(predictions=preds, references=refs)
cer = cer_metric.compute(predictions=preds, references=refs)
print(f"WER: {wer}, CER: {cer}")

100%|██████████| 667/667 [09:28<00:00,  1.17it/s]


Sample predictions:
Ref: це недорого якщо враховувати що 80 гонорару гурту мозгі йде на ліки від радикуліту для дяді ваді
Pred:  Це не дорого, якщо врахувати, що війсеміся пісотки вона рару, рурту, мозгій, де налікі відради кулі тут для дяді Ваді.
--------------------
Ref: з медицини а це так обовязкова була ваша присутність
Pred:  З медицин. А це таку бояського була ваше.
--------------------
Ref: то він вже почав виконувати умови контракту і саме під впливом цієї угоди за тиждень до того за тиждень до того
Pred:  Товін вже почав виконувати умову контракту. І саме підцепливом цієї уводи за тиждень до того, за тиждень до того.
--------------------
Ref: президентські вибори 2014 року
Pred:  президентські вибори 2019 року.
--------------------
Ref: не варто брати у дорослих дядь цукерки а й те що не варто брати у них лайки
Pred:  Не варто брата у доросла кдецюкерки. А йте, що не варто брата у них лайки.
--------------------
Ref: так на грудях є відеореєстратор значить синю лінію потім до